In [1]:
from pathlib import Path
import json

PROJECT = Path("/home/jovyan/project/georgia-bid-readiness")

config = {
    "solicitation_id": "commerce-ga-26-008",
    "solicitation_number": "26-008",
    "title": "Hotel Feasibility Analysis",
    "agency": "City of Commerce, Georgia",
    "source_page": "https://www.commercega.gov/business/bid-opportunities",
    "folder_name": "26-008 Hotel Feasibility Analysis",

    # Preserve the original filename spelling used by the source.
    "original_filename": "RFP 26-008 Hotel Feasiblity Analysis.pdf",
    "expected_addenda": [1, 2],
}

config_path = PROJECT / "configs/commerce_hotel.json"
config_path.parent.mkdir(parents=True, exist_ok=True)

if config_path.exists():
    if json.loads(config_path.read_text(encoding="utf-8")) != config:
        raise RuntimeError("A different configuration already exists.")
else:
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

(PROJECT / "src/ingestion").mkdir(parents=True, exist_ok=True)

print("Configuration saved:", config_path)

Configuration saved: /home/jovyan/project/georgia-bid-readiness/configs/commerce_hotel.json


In [2]:
%%writefile /home/jovyan/project/georgia-bid-readiness/src/ingestion/commerce.py
from pathlib import Path
from html.parser import HTMLParser
from urllib.parse import urljoin, urlsplit, unquote
from datetime import datetime, timezone
import hashlib
import io
import json
import re
import time

import requests
from pypdf import PdfReader


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def save_json(path, data):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(data, indent=2), encoding="utf-8")
    temporary.replace(path)


class LinkCollector(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            href = dict(attrs).get("href")
            if href:
                self.links.append(href)


def ingest(project, config_path):
    project = Path(project).resolve()
    config_path = Path(config_path).resolve()
    config = json.loads(config_path.read_text(encoding="utf-8"))
    sid = config["solicitation_id"]

    if not re.fullmatch(r"[a-z0-9-]+", sid):
        raise ValueError("Invalid solicitation ID.")

    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    raw = project / "data/raw" / sid
    manifests = project / "data/processed/manifests"
    raw.mkdir(parents=True, exist_ok=True)
    manifests.mkdir(parents=True, exist_ok=True)

    manifest_path = manifests / f"{sid}_{run_id}.json"
    manifest = {
        "schema_version": 1,
        "ingestion_run_id": run_id,
        "started_at_utc": utc_now(),
        "solicitation_id": sid,
        "solicitation_number": config["solicitation_number"],
        "title": config["title"],
        "agency": config["agency"],
        "source_page": config["source_page"],
        "configuration": config,
        "expected_document_count": 1 + len(config["expected_addenda"]),
        "packet_download_complete": False,
        "status": "running",
        "records": [],
    }
    save_json(manifest_path, manifest)

    with requests.Session() as session:
        session.headers.update({
            "User-Agent": "GeorgiaBidReadiness/0.1 (academic document collection)"
        })

        try:
            listing = session.get(
                config["source_page"], timeout=(15, 60)
            )
            listing.raise_for_status()

            snapshot = raw / f"source_listing_{run_id}.html"
            snapshot.write_bytes(listing.content)
            manifest["source_listing_snapshot"] = (
                snapshot.relative_to(project).as_posix()
            )
            manifest["source_listing_sha256"] = hashlib.sha256(
                listing.content
            ).hexdigest()

            parser = LinkCollector()
            parser.feed(listing.text)
            documents = []
            seen = set()

            for href in parser.links:
                url = urljoin(listing.url, href)
                path = unquote(urlsplit(url).path)
                filename = path.rsplit("/", 1)[-1]

                if f"/{config['folder_name']}/" not in path:
                    continue
                if not filename.lower().endswith(".pdf") or url in seen:
                    continue
                seen.add(url)

                match = re.search(r"ADDENDUM\s*#?\s*(\d+)", filename, re.I)

                if filename == config["original_filename"]:
                    number = None
                    kind = "original"
                    document_id = f"{sid}-original"
                elif match:
                    number = int(match.group(1))
                    kind = "addendum"
                    document_id = f"{sid}-addendum-{number:02d}"
                else:
                    continue

                documents.append({
                    "document_id": document_id,
                    "solicitation_id": sid,
                    "agency": config["agency"],
                    "document_type": kind,
                    "amendment_number": number,
                    "source_filename": filename,
                    "source_url": url,
                    "source_page": config["source_page"],
                })

            documents.sort(key=lambda d: d["amendment_number"] or 0)
            originals = [d for d in documents if d["document_type"] == "original"]
            numbers = [
                d["amendment_number"] for d in documents
                if d["document_type"] == "addendum"
            ]

            manifest["discovered_documents"] = documents
            save_json(manifest_path, manifest)

            if (
                len(originals) != 1
                or sorted(numbers) != sorted(config["expected_addenda"])
            ):
                raise ValueError(
                    f"Unexpected listing: {len(originals)} originals; "
                    f"amendments {numbers}. Review the source."
                )

        except Exception as error:
            manifest.update({
                "status": "failed",
                "error": str(error),
                "completed_at_utc": utc_now(),
            })
            save_json(manifest_path, manifest)
            raise

        for document in documents:
            record = {
                **document,
                "attempted_at_utc": utc_now(),
                "status": "pending",
            }

            try:
                response = session.get(
                    document["source_url"], timeout=(15, 90)
                )
                record.update({
                    "http_status": response.status_code,
                    "resolved_url": response.url,
                    "content_type": response.headers.get("Content-Type"),
                })
                response.raise_for_status()
                content = response.content

                if b"%PDF-" not in content[:1024]:
                    raise ValueError("Response has no PDF header.")

                reader = PdfReader(io.BytesIO(content))
                if reader.is_encrypted:
                    raise ValueError("Encrypted PDF requires review.")

                pages = len(reader.pages)
                if pages == 0:
                    raise ValueError("PDF contains no pages.")

                digest = hashlib.sha256(content).hexdigest()
                destination = raw / f"{document['document_id']}__{digest}.pdf"

                if destination.exists():
                    stored_hash = hashlib.sha256(
                        destination.read_bytes()
                    ).hexdigest()
                    if stored_hash != digest:
                        raise ValueError("Stored file failed its hash check.")
                    action = "reused_identical_file"
                else:
                    temporary = destination.with_suffix(".part")
                    temporary.write_bytes(content)
                    temporary.replace(destination)
                    action = "saved_new_version"

                record.update({
                    "status": "ok",
                    "retrieved_at_utc": utc_now(),
                    "sha256": digest,
                    "size_bytes": len(content),
                    "page_count": pages,
                    "local_path": destination.relative_to(project).as_posix(),
                    "storage_action": action,
                })
                print(
                    f"OK | {document['document_id']} | "
                    f"{pages} pages | {action}",
                    flush=True,
                )

            except Exception as error:
                record.update({
                    "status": "failed",
                    "error_type": type(error).__name__,
                    "error": str(error),
                })
                print(
                    f"FAILED | {document['document_id']} | {error}",
                    flush=True,
                )

            manifest["records"].append(record)
            save_json(manifest_path, manifest)
            time.sleep(1)

    successful = [r for r in manifest["records"] if r["status"] == "ok"]
    failed = [r for r in manifest["records"] if r["status"] == "failed"]
    complete = (
        len(successful) == manifest["expected_document_count"]
        and not failed
    )

    manifest.update({
        "completed_at_utc": utc_now(),
        "successful_document_count": len(successful),
        "failed_document_count": len(failed),
        "total_pages": sum(r["page_count"] for r in successful),
        "packet_download_complete": complete,
        "status": "ok" if complete else "partial_failure",
    })
    save_json(manifest_path, manifest)
    return manifest_path, manifest

Writing /home/jovyan/project/georgia-bid-readiness/src/ingestion/commerce.py


In [3]:
import importlib.util

module_path = PROJECT / "src/ingestion/commerce.py"
spec = importlib.util.spec_from_file_location(
    "commerce_ingestion", module_path
)
commerce = importlib.util.module_from_spec(spec)
spec.loader.exec_module(commerce)

manifest_path, result = commerce.ingest(PROJECT, config_path)

print("\n--- INGESTION SUMMARY ---")
print("Successful PDFs:", result["successful_document_count"])
print("Failed downloads:", result["failed_document_count"])
print("Total pages:", result["total_pages"])
print("Expected packet downloaded:", result["packet_download_complete"])
print("Manifest:", manifest_path)

OK | commerce-ga-26-008-original | 17 pages | saved_new_version
OK | commerce-ga-26-008-addendum-01 | 1 pages | saved_new_version
OK | commerce-ga-26-008-addendum-02 | 1 pages | saved_new_version

--- INGESTION SUMMARY ---
Successful PDFs: 3
Failed downloads: 0
Total pages: 19
Expected packet downloaded: True
Manifest: /home/jovyan/project/georgia-bid-readiness/data/processed/manifests/commerce-ga-26-008_20261019T235516670054Z.json


In [4]:
%%writefile /home/jovyan/project/georgia-bid-readiness/src/extraction/pdf_text.py
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata as metadata
import json
import re

from pypdf import PdfReader


def extract_packet(project, manifest_path, low_text_threshold=100):
    project = Path(project).resolve()
    manifest_path = Path(manifest_path).resolve()
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

    if not manifest.get("packet_download_complete"):
        raise ValueError("The input packet has incomplete downloads.")

    sid = manifest["solicitation_id"]
    if not re.fullmatch(r"[a-z0-9-]+", sid):
        raise ValueError("Invalid solicitation ID.")

    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    output = project / "data/processed/text" / sid / run_id
    output.mkdir(parents=True, exist_ok=False)

    extractor_version = metadata.version("pypdf")
    pages_path = output / "pages.jsonl"
    summary_path = output / "extraction_summary.json"

    summary = {
        "extraction_run_id": run_id,
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "solicitation_id": sid,
        "input_manifest": manifest_path.relative_to(project).as_posix(),
        "extractor": "pypdf",
        "extractor_version": extractor_version,
        "low_text_threshold": low_text_threshold,
        "expected_pages": manifest["total_pages"],
        "page_records_written": 0,
        "successful_pages": 0,
        "failed_pages": 0,
        "pages_needing_review": 0,
        "all_pages_extracted": False,
        "status": "running",
        "documents": [],
    }

    def save_summary():
        temporary = summary_path.with_suffix(".tmp")
        temporary.write_text(
            json.dumps(summary, indent=2), encoding="utf-8"
        )
        temporary.replace(summary_path)

    save_summary()

    with pages_path.open("w", encoding="utf-8") as page_file:
        for document in manifest["records"]:
            document_id = document["document_id"]
            records = []

            try:
                if document["status"] != "ok":
                    raise ValueError("Document was not downloaded successfully.")

                pdf_path = (project / document["local_path"]).resolve()
                if not pdf_path.is_relative_to(project):
                    raise ValueError("PDF path is outside the project.")

                digest = hashlib.sha256(pdf_path.read_bytes()).hexdigest()
                if digest != document["sha256"]:
                    raise ValueError("PDF hash differs from the manifest.")

                reader = PdfReader(pdf_path)
                if reader.is_encrypted:
                    raise ValueError("Encrypted PDF requires review.")

                if len(reader.pages) != document["page_count"]:
                    raise ValueError("PDF page count differs from the manifest.")

                for number, page in enumerate(reader.pages, start=1):
                    record = {
                        "extraction_run_id": run_id,
                        "document_id": document_id,
                        "solicitation_id": sid,
                        "document_sha256": digest,
                        "source_url": document["source_url"],
                        "source_pdf": document["local_path"],
                        "page_number": number,
                        "extraction_method": "pypdf",
                        "extractor_version": extractor_version,
                    }

                    try:
                        raw_text = page.extract_text() or ""
                        clean_text = raw_text.replace("\r\n", "\n")
                        clean_text = clean_text.replace("\r", "\n")
                        clean_text = re.sub(r"[ \t]+", " ", clean_text)
                        clean_text = "\n".join(
                            line.strip() for line in clean_text.splitlines()
                        ).strip()

                        characters = len(re.sub(r"\s", "", raw_text))
                        flags = []

                        if not raw_text.strip():
                            flags.append("empty_text")
                        elif characters < low_text_threshold:
                            flags.append("low_text")

                        record.update({
                            "status": "ok",
                            "raw_text": raw_text,
                            "clean_text": clean_text,
                            "character_count": len(raw_text),
                            "non_whitespace_characters": characters,
                            "quality_flags": flags,
                            "needs_review": bool(flags),
                        })

                    except Exception as error:
                        record.update({
                            "status": "failed",
                            "raw_text": None,
                            "clean_text": None,
                            "character_count": None,
                            "non_whitespace_characters": None,
                            "quality_flags": ["extraction_error"],
                            "needs_review": True,
                            "error": str(error),
                        })

                    page_file.write(
                        json.dumps(record, ensure_ascii=False) + "\n"
                    )
                    page_file.flush()
                    records.append(record)

                    summary["page_records_written"] += 1
                    summary["successful_pages"] += int(record["status"] == "ok")
                    summary["failed_pages"] += int(record["status"] == "failed")
                    summary["pages_needing_review"] += int(record["needs_review"])

                failures = sum(r["status"] == "failed" for r in records)
                flagged = sum(r["needs_review"] for r in records)

                summary["documents"].append({
                    "document_id": document_id,
                    "status": "partial_failure" if failures else "ok",
                    "page_count": len(records),
                    "failed_pages": failures,
                    "pages_needing_review": flagged,
                })

                print(
                    f"{document_id}: {len(records)} pages, "
                    f"{flagged} flagged for review",
                    flush=True,
                )

            except Exception as error:
                summary["documents"].append({
                    "document_id": document_id,
                    "status": "failed",
                    "error": str(error),
                })
                print(f"FAILED | {document_id} | {error}", flush=True)

            save_summary()

    summary["all_pages_extracted"] = (
        summary["successful_pages"] == summary["expected_pages"]
        and all(d["status"] == "ok" for d in summary["documents"])
    )
    summary["status"] = (
        "ok" if summary["all_pages_extracted"] else "partial_failure"
    )
    summary["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
    save_summary()

    return output, summary

Writing /home/jovyan/project/georgia-bid-readiness/src/extraction/pdf_text.py


In [5]:
from pathlib import Path
import importlib.util
import json

PROJECT = Path("/home/jovyan/project/georgia-bid-readiness")

hotel_manifest = PROJECT / (
    "data/processed/manifests/"
    "commerce-ga-26-008_20261019T235516670054Z.json"
)

module_path = PROJECT / "src/extraction/pdf_text.py"
spec = importlib.util.spec_from_file_location("pdf_text", module_path)
extractor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(extractor)

hotel_output, hotel_summary = extractor.extract_packet(
    PROJECT, hotel_manifest
)

print("\n--- EXTRACTION SUMMARY ---")
for key in [
    "expected_pages",
    "page_records_written",
    "successful_pages",
    "failed_pages",
    "pages_needing_review",
    "all_pages_extracted",
]:
    print(f"{key}: {hotel_summary[key]}")

print("Output:", hotel_output)

commerce-ga-26-008-original: 17 pages, 1 flagged for review
commerce-ga-26-008-addendum-01: 1 pages, 0 flagged for review
commerce-ga-26-008-addendum-02: 1 pages, 0 flagged for review

--- EXTRACTION SUMMARY ---
expected_pages: 19
page_records_written: 19
successful_pages: 19
failed_pages: 0
pages_needing_review: 1
all_pages_extracted: True
Output: /home/jovyan/project/georgia-bid-readiness/data/processed/text/commerce-ga-26-008/20261019T235650287784Z


In [6]:
hotel_pages = [
    json.loads(line)
    for line in (hotel_output / "pages.jsonl")
    .read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("--- FIRST-PAGE PREVIEWS ---")
for page in hotel_pages:
    if page["page_number"] == 1:
        print("\n", page["document_id"])
        print((page["raw_text"] or "[No extracted text]")[:600])

print("\n--- FLAGGED PAGES ---")
flagged = [page for page in hotel_pages if page["needs_review"]]

if not flagged:
    print("No pages flagged by the initial screening.")

for page in flagged:
    print(
        page["document_id"],
        "| PDF page", page["page_number"],
        "|", page["quality_flags"],
    )
    print("Extracted text:", repr((page["raw_text"] or "")[:300]))

--- FIRST-PAGE PREVIEWS ---

 commerce-ga-26-008-original
1  
 RFP 26-008 
Hotel Feasibility Analysis  
City of Commerce  
P.O. Box 348  
110 State Street  
Commerce, GA 30529  
Phone: (706) 335-1893  
Email: bids@commercega.gov  
    
      
  
SUBJECT:   26-008 Request for Proposal for Hotel Feasibility Analysis  
  
The City of Commerce, Georgia is soliciting competitive sealed proposals from qualified hotel 
consulting firms to perform a feasibility analysis for hotel development within the community.    
  
Attached hereto are the general conditions, technical specifications, and submittal format:  
  
The written requirements contained in this 

 commerce-ga-26-008-addendum-01
 
ADDENDUM #1 
RFP 26-008 
 
 
HOTEL FEASIBLITY ANALYSIS 
 
 
This addendum is being issued to address a question for RFP 26-008: 
 
1. Will performance and bid bonds be required? 
            Please see Section 1.20 BONDING on page 3 of the RFP. 
2. What is the last day for addenda to be issued? 
The last 

In [7]:
from pathlib import Path
from pypdf import PdfReader, PdfWriter
from IPython.display import display, FileLink
import os

page_to_review = next(
    page for page in hotel_pages
    if page["document_id"] == "commerce-ga-26-008-original"
    and page["page_number"] == 12
)

source_pdf = PROJECT / page_to_review["source_pdf"]
review_folder = PROJECT / "reports/page_review"
review_folder.mkdir(parents=True, exist_ok=True)

review_pdf = review_folder / "commerce_hotel_original_page_12.pdf"

reader = PdfReader(source_pdf)
writer = PdfWriter()
writer.add_page(reader.pages[11])

with review_pdf.open("wb") as file:
    writer.write(file)

print("Extracted text:")
print(repr(page_to_review["raw_text"]))
print("\nReview PDF:", review_pdf)

display(FileLink(os.path.relpath(review_pdf, Path.cwd())))

Extracted text:
'12  \n RFP 26-008 \nHotel Feasibility Analysis  \n \n \n  \n  \n \n  \n \n  \nRequired Forms for Submission  '

Review PDF: /home/jovyan/project/georgia-bid-readiness/reports/page_review/commerce_hotel_original_page_12.pdf


/home/jovyan/project/georgia-bid-readiness/reports/page_review/commerce_hotel_original_page_12.pdf

In [8]:
from datetime import datetime, timezone
import json

page = next(
    p for p in hotel_pages
    if p["document_id"] == "commerce-ga-26-008-original"
    and p["page_number"] == 12
)

review = {
    "extraction_run_id": page["extraction_run_id"],
    "document_id": page["document_id"],
    "document_sha256": page["document_sha256"],
    "page_number": page["page_number"],
    "original_quality_flags": page["quality_flags"],
    "review_status": "resolved",
    "resolution": "legitimate_low_text_page",
    "ocr_required": False,
    "reviewed_by": "Ibrahim Shittu",
    "reviewed_at_utc": datetime.now(timezone.utc).isoformat(),
    "review_method": "visual inspection of PDF page",
    "notes": (
        "Reviewer confirmed a required-forms divider page with "
        "RFP 26-008 and Hotel Feasibility Analysis identification. "
        "No additional substantive content was visible. "
        "Low text is expected; OCR is not required."
    ),
}

review_path = hotel_output / "page_reviews.jsonl"

existing = []
if review_path.exists():
    existing = [
        json.loads(line)
        for line in review_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

already_recorded = any(
    r["extraction_run_id"] == review["extraction_run_id"]
    and r["document_id"] == review["document_id"]
    and r["page_number"] == review["page_number"]
    and r.get("review_status") == "resolved"
    and r.get("resolution") == review["resolution"]
    and r.get("ocr_required") is False
    and r.get("reviewed_by") == review["reviewed_by"]
    for r in existing
)

if already_recorded:
    print("This resolution is already recorded; no duplicate added.")
else:
    with review_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(review, ensure_ascii=False) + "\n")
    print("Page 12 review saved: resolved; OCR not required.")

print("Review file:", review_path)

Page 12 review saved: resolved; OCR not required.
Review file: /home/jovyan/project/georgia-bid-readiness/data/processed/text/commerce-ga-26-008/20261019T235650287784Z/page_reviews.jsonl


In [9]:
(PROJECT / "src/transformation").mkdir(parents=True, exist_ok=True)

In [10]:
%%writefile /home/jovyan/project/georgia-bid-readiness/src/transformation/warehouse_loader.py
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import psycopg
from psycopg import sql
from psycopg.types.json import Jsonb


def read_jsonl(path):
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def checked_path(project, relative):
    path = (project / relative).resolve()
    if not path.is_relative_to(project):
        raise ValueError("Artifact path is outside the project.")
    return path


def run_time(run_id):
    return datetime.strptime(
        run_id, "%Y%m%dT%H%M%S%fZ"
    ).replace(tzinfo=timezone.utc)


def insert_or_verify(cursor, table, values):
    """Preserve identical records; reject conflicting existing records."""
    cursor.execute(
        sql.SQL("""
            INSERT INTO {} ({}) VALUES ({})
            ON CONFLICT DO NOTHING
        """).format(
            sql.Identifier("core", table),
            sql.SQL(", ").join(map(sql.Identifier, values)),
            sql.SQL(", ").join(sql.Placeholder() for _ in values),
        ),
        list(values.values()),
    )

    if cursor.rowcount == 1:
        return

    conditions = sql.SQL(" AND ").join(
        sql.SQL("{} IS NOT DISTINCT FROM %s").format(sql.Identifier(key))
        for key in values
    )
    cursor.execute(
        sql.SQL("SELECT 1 FROM {} WHERE {} LIMIT 1").format(
            sql.Identifier("core", table), conditions
        ),
        list(values.values()),
    )

    if cursor.fetchone() is None:
        raise ValueError(
            f"Conflicting existing record in core.{table}; "
            "the transaction will be rolled back."
        )


def load_packet(project, extraction_dir, db_settings):
    project = Path(project).resolve()
    extraction_dir = Path(extraction_dir).resolve()

    if not extraction_dir.is_relative_to(project):
        raise ValueError("Extraction directory is outside the project.")

    summary_path = extraction_dir / "extraction_summary.json"
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    manifest_path = checked_path(project, summary["input_manifest"])
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

    # This first reusable loader accepts complete runs.
    if not manifest.get("packet_download_complete"):
        raise ValueError("Ingestion is incomplete.")
    if not summary.get("all_pages_extracted"):
        raise ValueError("Extraction is incomplete.")

    sid = manifest["solicitation_id"]
    ingestion_id = manifest["ingestion_run_id"]
    extraction_id = summary["extraction_run_id"]
    documents = manifest["records"]
    pages = read_jsonl(extraction_dir / "pages.jsonl")

    review_path = extraction_dir / "page_reviews.jsonl"
    reviews = read_jsonl(review_path) if review_path.exists() else []

    if len(documents) != manifest["expected_document_count"]:
        raise ValueError("Unexpected document count.")

    by_document = {d["document_id"]: d for d in documents}
    if len(by_document) != len(documents):
        raise ValueError("Duplicate document IDs in the manifest.")

    for document in documents:
        if document["status"] != "ok" or document["solicitation_id"] != sid:
            raise ValueError("Invalid document record.")

        source = checked_path(project, document["local_path"])
        if hashlib.sha256(source.read_bytes()).hexdigest() != document["sha256"]:
            raise ValueError(f"Source hash mismatch: {document['document_id']}")

    page_lookup = {}
    for page in pages:
        key = (page["document_id"], page["page_number"])
        document = by_document.get(page["document_id"])

        if key in page_lookup:
            raise ValueError("Duplicate page record.")
        if (
            document is None
            or page["extraction_run_id"] != extraction_id
            or page["solicitation_id"] != sid
            or page["document_sha256"] != document["sha256"]
            or page["status"] != "ok"
        ):
            raise ValueError("Page record does not match the input packet.")

        page_lookup[key] = page

    for document in documents:
        actual = sorted(
            p["page_number"] for p in pages
            if p["document_id"] == document["document_id"]
        )
        if actual != list(range(1, document["page_count"] + 1)):
            raise ValueError("Missing or unexpected page numbers.")

    if len(pages) != summary["expected_pages"]:
        raise ValueError("Page count differs from extraction summary.")

    for review in reviews:
        page = page_lookup.get((review["document_id"], review["page_number"]))
        if (
            page is None
            or review["extraction_run_id"] != extraction_id
            or review["document_sha256"] != page["document_sha256"]
        ):
            raise ValueError("Review does not match its extracted page.")

    with psycopg.connect(**db_settings) as connection:
        with connection.cursor() as cursor:
            def put(table, values):
                insert_or_verify(cursor, table, values)

            put("solicitations", {
                "solicitation_id": sid,
                "agency_name": manifest["agency"],
                "solicitation_number": manifest["solicitation_number"],
                "title": manifest["title"],
                "source_page_url": manifest["source_page"],
            })

            put("pipeline_runs", {
                "run_id": ingestion_id,
                "run_type": "ingestion",
                "status": "ok",
                "started_at": manifest.get("started_at_utc") or run_time(ingestion_id),
                "completed_at": manifest["completed_at_utc"],
                "artifact_path": manifest_path.relative_to(project).as_posix(),
                "metadata": Jsonb({
                    "expected_document_count": manifest["expected_document_count"],
                    "source_listing_sha256": manifest["source_listing_sha256"],
                }),
            })

            for document in documents:
                put("documents", {
                    "document_id": document["document_id"],
                    "solicitation_id": sid,
                    "document_type": document["document_type"],
                    "amendment_number": document["amendment_number"],
                })
                put("document_versions", {
                    "document_id": document["document_id"],
                    "sha256": document["sha256"],
                    "local_path": document["local_path"],
                    "size_bytes": document["size_bytes"],
                    "page_count": document["page_count"],
                })
                put("document_observations", {
                    "run_id": ingestion_id,
                    "document_id": document["document_id"],
                    "version_sha256": document["sha256"],
                    "source_url": document["source_url"],
                    "source_filename": document["source_filename"],
                    "resolved_url": document.get("resolved_url"),
                    "attempted_at": document["attempted_at_utc"],
                    "retrieved_at": document["retrieved_at_utc"],
                    "http_status": document.get("http_status"),
                    "status": "ok",
                    "storage_action": document["storage_action"],
                })

            put("pipeline_runs", {
                "run_id": extraction_id,
                "run_type": "extraction",
                "parent_run_id": ingestion_id,
                "status": "ok",
                "started_at": summary.get("started_at_utc") or run_time(extraction_id),
                "completed_at": summary["completed_at_utc"],
                "artifact_path": summary_path.relative_to(project).as_posix(),
                "metadata": Jsonb({
                    "extractor": summary["extractor"],
                    "extractor_version": summary["extractor_version"],
                    "low_text_threshold": summary["low_text_threshold"],
                    "expected_pages": summary["expected_pages"],
                }),
            })

            page_fields = [
                "extraction_run_id", "document_id", "document_sha256",
                "page_number", "extraction_method", "extractor_version",
                "status", "raw_text", "clean_text", "character_count",
                "non_whitespace_characters", "needs_review",
            ]

            for page in pages:
                values = {key: page[key] for key in page_fields}
                values["quality_flags"] = Jsonb(page["quality_flags"])
                values["error_message"] = page.get("error")
                put("page_extractions", values)

            for review in reviews:
                values = {
                    key: review[key] for key in [
                        "extraction_run_id", "document_id", "page_number",
                        "review_status", "resolution", "ocr_required",
                        "reviewed_by", "review_method", "notes",
                    ]
                }
                values["reviewed_at"] = review["reviewed_at_utc"]
                put("page_reviews", values)

    return {
        "solicitation_id": sid,
        "input_documents": len(documents),
        "input_pages": len(pages),
        "input_reviews": len(reviews),
        "ingestion_run_id": ingestion_id,
        "extraction_run_id": extraction_id,
        "status": "committed",
    }

Writing /home/jovyan/project/georgia-bid-readiness/src/transformation/warehouse_loader.py


In [11]:
from pathlib import Path
import importlib.util

PROJECT = Path("/home/jovyan/project/georgia-bid-readiness")

DB_SETTINGS = {
    "host": "/home/jovyan/.gbr-pg-socket",
    "dbname": "georgia_bid_readiness",
    "user": "jovyan",
    "connect_timeout": 10,
}

hotel_output = PROJECT / (
    "data/processed/text/commerce-ga-26-008/"
    "20261019T235650287784Z"
)

module_path = PROJECT / "src/transformation/warehouse_loader.py"
spec = importlib.util.spec_from_file_location("warehouse_loader", module_path)
loader = importlib.util.module_from_spec(spec)
spec.loader.exec_module(loader)

result = loader.load_packet(PROJECT, hotel_output, DB_SETTINGS)
print(result)

{'solicitation_id': 'commerce-ga-26-008', 'input_documents': 3, 'input_pages': 19, 'input_reviews': 1, 'ingestion_run_id': '20261019T235516670054Z', 'extraction_run_id': '20261019T235650287784Z', 'status': 'committed'}


In [12]:
import psycopg
from psycopg import sql

tables = [
    "solicitations",
    "documents",
    "document_versions",
    "pipeline_runs",
    "document_observations",
    "page_extractions",
    "page_reviews",
]

def get_counts():
    counts = {}
    with psycopg.connect(**DB_SETTINGS) as connection:
        with connection.cursor() as cursor:
            for table in tables:
                cursor.execute(
                    sql.SQL("SELECT COUNT(*) FROM {}").format(
                        sql.Identifier("core", table)
                    )
                )
                counts[table] = cursor.fetchone()[0]
    return counts

before = get_counts()

# Repeat exactly the same load.
loader.load_packet(PROJECT, hotel_output, DB_SETTINGS)

after = get_counts()
assert before == after, "Repeat load changed the row counts."

for table, count in after.items():
    print(f"{table}: {count}")

print("\nRepeat-load check passed: no duplicates added.")

solicitations: 2
documents: 15
document_versions: 15
pipeline_runs: 5
document_observations: 27
page_extractions: 47
page_reviews: 2

Repeat-load check passed: no duplicates added.


In [13]:
from datetime import datetime, timezone
import json
import psycopg
from psycopg.rows import dict_row

# The existing SQL already accepts solicitation-specific parameters.
sql_path = PROJECT / "warehouse/tests/001_commerce_packet_quality.sql"
quality_sql = sql_path.read_text(encoding="utf-8")

scope = {
    "solicitation_id": "commerce-ga-26-008",
    "extraction_run_id": "20261019T235650287784Z",
    "expected_addenda": 2,
}

with psycopg.connect(
    **DB_SETTINGS,
    row_factory=dict_row,
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(quality_sql, scope)
        results = cursor.fetchall()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
report_folder = PROJECT / "reports/data_quality"
report_folder.mkdir(parents=True, exist_ok=True)

report = {
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "scope": scope,
    "sql_file": sql_path.relative_to(PROJECT).as_posix(),
    "all_checks_passed": all(row["passed"] for row in results),
    "results": results,
}

report_path = report_folder / f"commerce_hotel_quality_{stamp}.json"
report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

for row in results:
    print(
        f"{'PASS' if row['passed'] else 'FAIL'} | "
        f"{row['check_name']} | failures={row['failure_count']}"
    )

print("\nAll checks passed:", report["all_checks_passed"])
print("Report saved:", report_path)

PASS | 01_run_and_packet_present | failures=0
PASS | 02_original_count | failures=0
PASS | 03_amendment_sequence | failures=0
PASS | 04_document_page_coverage | failures=0
PASS | 05_page_numbering | failures=0
PASS | 06_extraction_failures | failures=0
PASS | 07_unresolved_review_flags | failures=0
PASS | 08_processing_lineage | failures=0

All checks passed: True
Report saved: /home/jovyan/project/georgia-bid-readiness/reports/data_quality/commerce_hotel_quality_20261020T000528597997Z.json
